# Task 3: Support Vector Machine (SVM) Classification

Train SVM models with **linear** and **RBF** kernels and compare their performance with the logistic regression baseline.

**Goal**: Evaluate kernel selection impact on classification accuracy.

## 1. Initialize Project Environment

In [1]:
import logging
import sys
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict

import pandas as pd
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

# Configure logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s"
)

print(f"Python {sys.version}")

Python 3.12.3 (main, Jan  8 2026, 11:30:50) [GCC 13.3.0]


## 2. Define Configuration Parameters

In [2]:
@dataclass
class TaskConfig:
    handle: str
    artifacts_dir: Path = Path("artifacts")
    random_state: int = 42

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["artifacts_dir"] = str(info["artifacts_dir"])
        return info


CONFIG = TaskConfig(handle="rbals")
CONFIG.describe()

{'handle': 'rbals', 'artifacts_dir': 'artifacts', 'random_state': 42}

## 3. Implement Core Functionality

In [3]:
def compare_svm_kernels(config: TaskConfig):
    # Load processed data
    X_train = pd.read_csv(config.artifacts_dir / "task1_X_train_scaled.csv")
    X_test = pd.read_csv(config.artifacts_dir / "task1_X_test_scaled.csv")
    y_train = pd.read_csv(config.artifacts_dir / "task1_y_train.csv").values.flatten()
    y_test = pd.read_csv(config.artifacts_dir / "task1_y_test.csv").values.flatten()

    results = []

    # Baseline: Load LogReg metrics
    logreg_metrics = pd.read_csv(
        config.artifacts_dir / "task2_logistic_regression_metrics.csv"
    ).iloc[0]
    results.append(
        {
            "Model": "Logistic Regression",
            "Kernel": "N/A",
            "Accuracy": logreg_metrics["accuracy"],
            "F1": logreg_metrics["f1_score"],
        }
    )

    # SVM Linear
    svm_linear = SVC(kernel="linear", random_state=config.random_state)
    svm_linear.fit(X_train, y_train)
    y_pred_lin = svm_linear.predict(X_test)
    results.append(
        {
            "Model": "SVM",
            "Kernel": "linear",
            "Accuracy": accuracy_score(y_test, y_pred_lin),
            "F1": f1_score(y_test, y_pred_lin),
        }
    )

    # SVM RBF
    svm_rbf = SVC(kernel="rbf", random_state=config.random_state)
    svm_rbf.fit(X_train, y_train)
    y_pred_rbf = svm_rbf.predict(X_test)
    results.append(
        {
            "Model": "SVM",
            "Kernel": "rbf",
            "Accuracy": accuracy_score(y_test, y_pred_rbf),
            "F1": f1_score(y_test, y_pred_rbf),
        }
    )

    results_df = pd.DataFrame(results)
    logging.info(f"Comparison Summary: \n{results_df}")

    return results_df


COMPARISON_DF = compare_svm_kernels(CONFIG)

2026-02-01 16:29:52,397 | INFO | Comparison Summary: 
                 Model  Kernel  Accuracy        F1
0  Logistic Regression     N/A  0.852941  0.909091
1                  SVM  linear  0.764706  0.851852
2                  SVM     rbf  0.882353  0.925926


## 4. Validate with Unit Tests

In [4]:
assert len(COMPARISON_DF) == 3, "Result count mismatch"
assert "Accuracy" in COMPARISON_DF.columns, "Metrics mismatch"
print("[OK] Validation passed.")

[OK] Validation passed.


## 5. Export Results

In [5]:
EXPORT_DIR = CONFIG.artifacts_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

out_file = EXPORT_DIR / "task3_svm_comparison.csv"
COMPARISON_DF.to_csv(out_file, index=False)
print(f"[OK] Comparison results saved to: {out_file.resolve()}")

[OK] Comparison results saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/08_ML_flower/assignments/artifacts/task3_svm_comparison.csv
